In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
from tqdm.auto import tqdm

## Test PL module

In [2]:
!export HDF5_USE_FILE_LOCKING=FALSE


In [3]:

import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)

LitAudioSSL = lightning.LitAudioSSL
## init config. Will be yaml eventually, but start as dict 

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_dualtask_resnet18_MatchedSpeechInNoiseDatasetBatched.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 
model = LitAudioSSL(config)

In [ ]:
model

LitAudioSSL(
  (audio_rep): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): ModelWithFrontEnd(
    (front_end): AudioToAudioRepresentation(
      (rep): AudioToCochleagram(
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling_op): SincWithKaiserWindow()
        (Cochleagram): Cochleagram(
          (compute_subbands): ComputeSubbands()
          (envelope_extraction): HilbertEnvelopeExtraction()
          (downsampling): SincWithKaiserWindow()
        )
      )
      (compression): ClippedGradPower(
        (compression_function): C

In [5]:
from jsinV3DataLoader_precombined_batched import MatchedSpeechInNoiseDatasetBatched


In [6]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)
trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 37.3 M | train
2 | ssl_loss        | Paired_Loss                | 0      | train
3 | multi_task_loss | jsinV3_multi_task_loss     | 0      | train
4 | metrics         | ModuleDict                 | 0      | train
----------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
dataset = MatchedSpeechInNoiseDatasetBatched(speech_h5_path=config['data']['val_speech_h5_path'],
                                                     noise_h5_path=config['data']['val_noise_h5_path'],
                                                     low_db=config['audio_transforms']['low_snr'],
                                                     high_db=config['audio_transforms']['high_snr'],
                                                     db_spl=config['audio_transforms']['dbspl'],
                                                     batch_size=config['hparas']['batch_size'],
                                                     target_keys=config['data'].get("target_keys", None),
                                                     )

In [15]:
audio, labels = dataset[1]

### Dev sudo coe

In [ ]:
from torchvision.models.resnet import resnet50
from robustness.audio_models import resnet50 as resnet50_robusntess
import torch.nn as nn 

class SSLAudioModelWMetamers(nn.Module):
    def __init__(self, projector_dims=[512, 512], proj_out_dim=2048, n_classes=794, supervised=False, **kwargs):
        super().__init__()
        self.supervised = supervised

        self.f = resnet50_robusntess()
        self.f.fc = nn.Identity()

        # projection head (Following exactly barlow twins offical repo)
        projector_dims = [proj_out_dim] + projector_dims
        layers = []
        for i in range(len(projector_dims) - 2):
            layers.append(
                nn.Linear(projector_dims[i], projector_dims[i + 1], bias=False)
            )
            layers.append(nn.BatchNorm1d(projector_dims[i + 1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(projector_dims[-2], projector_dims[-1], bias=False))
        self.g = nn.Sequential(*layers)
        if supervised:
            self.lin_cls = nn.Linear(proj_out_dim, n_classes)

    def forward(self, x):
        x_ = self.f(x)
        feature = torch.flatten(x_, start_dim=1)
        out = self.g(feature)
        if not self.supervised:
            return feature, out, None 
        else:
            logits = self.lin_cls(feature)
        return feature, out, logits

In [ ]:
import torch

In [ ]:
# model = SSLAudioModelWMetamers().cuda()

In [ ]:
ckpt_path = "model_checkpoints/pilot_ssl_audioset_resnet50/checkpoints/epoch=9-step=37500-v1.ckpt"
# checkpoint = torch.load(ckpt_path, weights_only=True)
# # update state dict 
# new_state_dict = {}
# for key,val in checkpoint['state_dict'].items():
#     new_key = key.split('model.model.')[-1] if key.startswith("model.model.") else key 
#     new_state_dict[new_key] = val
# # model.load_state_dict(new_state_dict, strict=False)

In [ ]:
# model.load_state_dict(checkpoint['state_dict'], strict=True)

<All keys matched successfully>

In [ ]:
import lightning_scripts.lightning_ssl as lightning 
import importlib
import yaml
import torch

importlib.reload(lightning)

config_path = "model_configs/pilot_ssl_audioset_resnet50.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4 
# config['hparas']['batch_size'] = config['hparas']['global_batch_size'] / 1 # total batch size / n gpus 
config['hparas']['batch_size'] = 64 // 1 # total batch size / n gpus 
ckpt_path = "model_checkpoints/pilot_ssl_audioset_resnet50/checkpoints/epoch=9-step=37500-v1.ckpt"

LitAudioSSL = lightning.LitAudioSSL
model = LitAudioSSL.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).eval().cuda()

In [ ]:
# model = model.cuda()

In [ ]:
# f = model.model.model.f

In [ ]:
model.model

ModelWithFrontEnd(
  (front_end): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): SSLAudioModelWMetamers(
    (f): ResNet(
      (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): SequentialWithArgs(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
   

In [ ]:
x = torch.rand(1,1,40000).cuda()
# f(x, with_latent=True)
model.model(x, with_latent=True)

(tensor([[2.3782, 0.5847, 1.5378,  ..., 1.1603, 1.2008, 0.7588]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor([[2.3782, 0.5847, 1.5378,  ..., 1.1603, 1.2008, 0.7588]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 {'input_after_preproc': tensor([[[[0.6600, 0.6591, 0.6593,  ..., 0.6582, 0.6619, 0.6693],
            [0.6594, 0.6598, 0.6596,  ..., 0.6589, 0.6597, 0.6689],
            [0.6588, 0.6606, 0.6599,  ..., 0.6594, 0.6578, 0.6689],
            ...,
            [0.2970, 0.2824, 0.2250,  ..., 0.2252, 0.2536, 0.2725],
            [0.2569, 0.2360, 0.1908,  ..., 0.1946, 0.1991, 0.2304],
            [0.1811, 0.1700, 0.1440,  ..., 0.1507, 0.1390, 0.1764]]]],
         device='cuda:0'),
  'conv1': tensor([[[[-0.1934, -0.4840, -0.5017,  ..., -0.4997, -0.5011, -0.5091],
            [ 0.0464, -0.2472, -0.2989,  ..., -0.3258, -0.3337, -0.3263],
            [-0.0434, -0.1679, -0.1895,  ..., -0.1962, -0.2206, -0.3024],
            ...,
            [-0.0856, -0.2461, -0.2175, 